## Ingestion Transazioni — Auto Loader

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType

trans_schema = StructType([
    StructField("trans_id",   StringType()),
    StructField("account_id", StringType()),
    StructField("date",       StringType()),
    StructField("type",       StringType()),
    StructField("operation",  StringType()),
    StructField("amount",     StringType()),
    StructField("balance",    StringType()),
    StructField("k_symbol",   StringType()),
    StructField("bank",       StringType()),
    StructField("account",    StringType()),
])

def ingest_transactions():
    query = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", True)
            .option("sep", ";")
            .schema(trans_schema)
            .load(TRANS_PATH)
            .withColumn("_ingestion_ts", F.current_timestamp())
            .writeStream
            .format("delta")
            .outputMode("append")
            .trigger(availableNow=True)
            .option("checkpointLocation", CHECKPOINT)
            .toTable(f"{CATALOG}.{BRONZE_SCHEMA}.transactions")
    )
    query.awaitTermination()

ingest_transactions()
spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.transactions").count()